In [ ]:
# Define coordinates as a list of [lon, lat]
coordinates_list = [
    [94.2604595275288, 29.773270070797224],  # 尼西村
    [100.112018, 25.653985], # 苍山
    [104.259534, 31.217034], # 德阳
    [91.911327, 29.181582], # 山南
    [92.603540, 28.944296], # 加查
    [94.361009, 29.648327], # 林芝
    [91.273398, 29.790422], # 拉萨
    [104.659162, 29.258789], # 自贡
    [115.897694, 33.007419], # 阜阳
    [116.337168, 31.356277], # 南岳
    [118.165609, 30.137470], # 黄山
    [102.269126, 27.962762], # 凉山
    [101.266320, 24.368124], # 景东
    [100.717270, 22.129849], # 西双版纳
    [109.041607, 24.310168], # 柳州
    # Add more coordinates here
]

# Image Size in pixels
pixel_size = 1024

image_collection_id = "COPERNICUS/S2_HARMONIZED"
bands_to_export = [
    "B2", # Blue
    "B3", # Green
    "B4", # Red
    "B8", # NIR
    "B11",# SWIR1
    "B12" # SWIR2
]
start_date = "2015-01-01"
end_date = "2026-12-31"
use_csplus = True
csplus_thresh = 0.8
cloud_filter_percentage = 20
export_folder = "EEExport"
export_scale = 10


In [ ]:
import ee
from ee.feature import Feature
from ee.featurecollection import FeatureCollection
from ee.geometry import Geometry
from ee.imagecollection import ImageCollection
from ee.image import Image
from ee.batch import Export
from ee.filter import Filter
from ee.join import Join

try:
    ee.Initialize(project="ee-yangluhao990714")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="ee-yangluhao990714")

In [ ]:
from platform import java_ver


for coords in coordinates_list:
    lon, lat = coords[0], coords[1]

    # Calculate buffer size in meters to get desired pixel dimensions
    buffer_size = (pixel_size * export_scale) / 2
    point = Geometry.Point([lon, lat])
    bbox = point.buffer(buffer_size).bounds()

    image_collection: ImageCollection = (ImageCollection(image_collection_id)
                        .filterBounds(bbox)
                        .filterDate(start_date, end_date))

    if cloud_filter_percentage is not None:
        image_collection = image_collection.filter(
            Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_filter_percentage)
        )

    if use_csplus:
        cs = ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
        join_filter = Filter.equals(leftField="system:index", rightField="system:index")
        join = Join.saveFirst(matchKey="csplus_mask")
        image_collection = join.apply(image_collection, cs, join_filter)
        def add_mask(img) -> Image:
            cs_img = ee.Image(img.get("csplus_mask"))
            return img.updateMask(cs_img.select("cs").gt(csplus_thresh))
        image_collection = image_collection.map(add_mask)

    for band_name in bands_to_export:
        time_series_cube = image_collection.select(band_name).toBands()

        collection_name_for_file = image_collection_id.replace("/", "_")
        file_name = f"{collection_name_for_file}_{band_name}_lon{lon:.4f}_lat{lat:.4f}"

        task = Export.image.toDrive(
            image=time_series_cube.toFloat(),
            description=file_name,
            folder=export_folder,
            fileNamePrefix=file_name,
            region=bbox,
            scale=export_scale,
            crs="EPSG:4326",
            maxPixels=1e10,
            # dimensions="1024x1024",
        )
        task.start()
